<html>
<head>
<meta charset="UTF-8">
<title>Software Village</title>
<style>
  body {
    margin: 0;
    background: #111;
    display: flex;
    justify-content: center;
    padding-top: 20px;
    font-family: Arial, sans-serif;
  }

  #sv-container {
    position: relative;
    width: 900px;
    height: 420px;
  }

  #sv-canvas {
    border: 1px solid #ccc;
    background: #000;
  }

  #sv-instructions {
    position: absolute;
    top: 10px;
    left: 10px;
    background: rgba(255,255,255,0.85);
    color: black;
    padding: 6px 10px;
    border-radius: 6px;
    font-size: 14px;
  }

  #sv-question-panel {
    position: absolute;
    bottom: 0;
    left: 0;
    right: 0;
    height: 80px;
    background: rgba(0,0,0,0.75);
    display: none;
    justify-content: space-around;
    align-items: center;
    color: white;
    font-size: 16px;
  }

  .sv-btn {
    padding: 10px 16px;
    border-radius: 6px;
    border: none;
    cursor: pointer;
    background: #444;
    color: white;
    font-size: 15px;
  }

  .sv-btn:hover {
    background: #666;
  }

  #sv-answer-box {
    position: absolute;
    bottom: 90px;
    left: 50%;
    transform: translateX(-50%);
    width: 70%;
    background: rgba(255,255,255,0.95);
    padding: 10px;
    border-radius: 8px;
    display: none;
    font-size: 15px;
    line-height: 1.4;
    color: black;
  }
</style>
</head>

<body>

<div id="sv-container">
  <canvas id="sv-canvas" width="900" height="400"></canvas>

  <div id="sv-instructions">
    Use WASD to move. Press E near a cottage.
  </div>

  <div id="sv-question-panel">
    <button class="sv-btn" data-type="what">What is this?</button>
    <button class="sv-btn" data-type="download">How to download?</button>
    <button class="sv-btn" data-type="use">How to use?</button>
  </div>

  <div id="sv-answer-box"></div>
</div>

<script>
// Prevent space scroll
window.addEventListener("keydown", e => {
  if (e.code === "Space") e.preventDefault();
});

// Setup
const canvas = document.getElementById("sv-canvas");
const ctx = canvas.getContext("2d");

const instructions = document.getElementById("sv-instructions");
const questionPanel = document.getElementById("sv-question-panel");
const answerBox = document.getElementById("sv-answer-box");

const WIDTH = canvas.width;
const HEIGHT = canvas.height;

// Images
const bg = new Image();
bg.src = "/images/projects/gamify/software_images.png";

const avatar = new Image();
avatar.src = "/images/projects/gamify/man.png";

// Player
const player = {
  x: WIDTH / 2,
  y: HEIGHT / 2,
  speed: 2.2,
  frozen: false,
  size: 22
};

// Cottages
const cottages = [
  { id: "jupyter", x: WIDTH/2 - 80, y: 60 },
  { id: "git", x: 100, y: HEIGHT/2 - 40 },
  { id: "pip", x: WIDTH - 200, y: HEIGHT/2 - 40 },
  { id: "python", x: 100, y: HEIGHT - 160 },
  { id: "ruby", x: WIDTH - 200, y: HEIGHT - 160 }
];

const COTTAGE_ANSWERS = {
  jupyter: {
    what: "Jupyter Notebook is an interactive coding environment.",
    download: "pip install notebook or install Anaconda.",
    use: "Run: jupyter notebook"
  },
  git: {
    what: "Git tracks code changes.",
    download: "Download from git-scm.com",
    use: "git init, git add, git commit"
  },
  pip: {
    what: "Pip installs Python packages.",
    download: "Comes with Python",
    use: "pip install package_name"
  },
  python: {
    what: "Python is a programming language.",
    download: "Download from python.org",
    use: "python file.py"
  },
  ruby: {
    what: "Ruby is used for web apps.",
    download: "Download from ruby-lang.org",
    use: "ruby file.rb"
  }
};

// Input
const keys = {};
document.addEventListener("keydown", e => keys[e.key] = true);
document.addEventListener("keyup", e => keys[e.key] = false);

// Interaction
let interacting = false;
let currentCottage = null;

// Arrow
let arrowAngle = 0;
let arrowActive = false;
let arrowX = 0;
let arrowY = 0;
let arrowVX = 0;
let arrowVY = 0;

function fireArrow() {
  arrowActive = true;
  arrowX = player.x + Math.cos(arrowAngle) * 12;
  arrowY = player.y - 20 + Math.sin(arrowAngle) * 12;
  arrowVX = Math.cos(arrowAngle) * 6;
  arrowVY = Math.sin(arrowAngle) * 6;
}

// Key controls
document.addEventListener("keydown", e => {

  // SPACE to exit after answering
  if (e.code === "Space" && answerBox.style.display === "block") {
    endInteraction();
    return;
  }

  if (!interacting) return;

  // Rotate with A/D
  if (e.key === "a" || e.key === "A") arrowAngle -= 0.1;
  if (e.key === "d" || e.key === "D") arrowAngle += 0.1;

  // Shoot
  if (e.key === " ") fireArrow();

  if (e.key === "Escape") endInteraction();
});

// Buttons
document.querySelectorAll(".sv-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    showAnswer(btn.dataset.type);
  });
});

// Show answer
function showAnswer(type) {
  answerBox.textContent = COTTAGE_ANSWERS[currentCottage][type];
  answerBox.style.display = "block";
}

// End interaction
function endInteraction() {
  interacting = false;
  arrowActive = false;
  questionPanel.style.display = "none";
  answerBox.style.display = "none";
  instructions.textContent = "Use WASD to move. Press E near a cottage.";
  player.frozen = false;
}

// Detect cottage
function nearCottage() {
  for (let c of cottages) {
    if (Math.hypot(player.x - c.x, player.y - c.y) < 50) return c.id;
  }
  return null;
}

// Enter interaction
document.addEventListener("keydown", e => {
  if (e.key !== "e" && e.key !== "E") return;

  const c = nearCottage();
  if (!c) return;

  interacting = true;
  currentCottage = c;
  player.frozen = true;

  questionPanel.style.display = "flex";
  answerBox.style.display = "none";

  instructions.textContent = "Use A/D to aim, SPACE to shoot.";
});

// Arrow update
function updateArrow() {
  if (!arrowActive) return;

  arrowX += arrowVX;
  arrowY += arrowVY;

  const buttons = document.querySelectorAll(".sv-btn");
  const rectCanvas = canvas.getBoundingClientRect();

  buttons.forEach(btn => {
    const rect = btn.getBoundingClientRect();
    const bx1 = rect.left - rectCanvas.left;
    const by1 = rect.top - rectCanvas.top;
    const bx2 = bx1 + rect.width;
    const by2 = by1 + rect.height;

    if (arrowX > bx1 && arrowX < bx2 && arrowY > by1 && arrowY < by2) {
      arrowActive = false;
      showAnswer(btn.dataset.type);
    }
  });

  if (arrowX < 0 || arrowX > WIDTH || arrowY < 0 || arrowY > HEIGHT) {
    arrowActive = false;
  }
}

// Draw
function draw() {
  ctx.clearRect(0, 0, WIDTH, HEIGHT);

  ctx.drawImage(bg, 0, 0, WIDTH, HEIGHT);

  // Player
  ctx.beginPath();
  ctx.arc(player.x, player.y, player.size + 6, 0, Math.PI * 2);
  ctx.fillStyle = "rgba(0,0,0,0.6)";
  ctx.fill();

  ctx.drawImage(avatar, player.x - 20, player.y - 20, 40, 40);

  // Cottages (only when NOT interacting)
  if (!interacting) {
    ctx.strokeStyle = "black";
    ctx.lineWidth = 2;
    cottages.forEach(c => {
      ctx.strokeRect(c.x - 20, c.y - 20, 40, 40);
    });
  }

  // Arrow
  if (interacting) {
    ctx.save();
    ctx.translate(player.x, player.y - 20);
    ctx.rotate(arrowAngle);

    ctx.beginPath();
    ctx.moveTo(0, -12);
    ctx.lineTo(6, 6);
    ctx.lineTo(-6, 6);
    ctx.closePath();
    ctx.fillStyle = "white";
    ctx.fill();

    ctx.restore();
  }

  // Projectile
  if (arrowActive) {
    ctx.beginPath();
    ctx.arc(arrowX, arrowY, 4, 0, Math.PI * 2);
    ctx.fillStyle = "white";
    ctx.fill();
  }
}

// Loop
function loop() {
  if (!player.frozen) {
    if (keys["w"] || keys["W"]) player.y -= player.speed;
    if (keys["s"] || keys["S"]) player.y += player.speed;
    if (keys["a"] || keys["A"]) player.x -= player.speed;
    if (keys["d"] || keys["D"]) player.x += player.speed;
  }

  updateArrow();
  draw();
  requestAnimationFrame(loop);
}

loop();
</script>

</body>
</html>

## 🔧 Tool Setup Troubleshooting Guide

Use this page if something is not working.  
Each section is independent — jump directly to the area you are stuck. 


```mermaid
flowchart TD
    %% GitHub Sources
    subgraph GitHub_Pages[GitHub: Open-Coding-Society/pages]
        A[Repo: pages]:::repo
    end

    subgraph GitHub_Template[GitHub: Open-Coding-Society/student]
        T[Template Repo: student]:::repo
    end

    subgraph GitHub_Student[GitHub: jm1021/student]
        B[Repo: student]:::repo
    end

    %% Local Computer
    subgraph Local[Local Computer]
        subgraph opencs_dir[opencs/ directory]
            C[pages/]:::local
            Ccmd[VSCode Prep<br/><br/>./scripts/venv.sh<br/>source venv/bin/activate<br/>code .]:::cmd
        end
        subgraph user_dir[jm1021/ directory]
            D[student/]:::local
            Dcmd[VSCode Prep<br/><br/>./scripts/venv.sh<br/>source venv/bin/activate<br/>code .]:::cmd
        end
    end

    %% Arrows: cloning
    A -.->|clone/pull only| C
    B <--> |clone, pull & push| D

    %% Arrows: template relationship
    T -.->|template→created| B

    %% Arrows: commands
    C --> Ccmd
    D <--> Dcmd

```

---

## GitHub Commit / Config Recovery

Use these commands if git commit is failing

✅ **Expectation**  
You have a GitHub username + email

```bash
git config --list` # shows your GitHub username + email.  
```

❌ **If not personalized, run to match your credentials:**  

```bash
git config --global user.name "jm1021"        # change to your GitHub ID
git config --global user.email "jm1021@gmail.com"  # change to your Email
```

Or run script that does the same thing from your project directory

```bash
./scripts/activate.sh # help setup git config options
```

---

## Directory + Clone Recovery

✅ Expectation: Project directory is setup correctly

You can cd into your personal directory, and an ls shows your repo folder (ex: student).

- In VSCode open this file (tool_setup-trouble.ipynb), run the script with comment "Check for presence of repo and virtual environment".
- Or step through commands in terminal to aid in your comprehension.

### Navigate

```bash
cd ~/jm1021 # change jm1021 to your user directory name
```

❌ If cd fails, run:

```bash
mkdir ~/jm1021
cd ~/jm1021
```

### Check for repo folder

```bash
ls # lshould show "student"
```

❌ If missing, run:

```bash
git clone https://github.com/jm1021/student.git # change to personal location of repo
```

### Move into repo folder

```bash
cd ~/jm1021/student
ls # see listing of project files
```

❌ If no directory or no file list, consider gap and discuss it with team member, then TA, and then Instructor

---

In [ ]:
%%script bash

# Location of student repo, change to match your personal setup
repoLocation=~/jm1021/student

# Check for presence of repo and virtual environment 
commands=(
    
  "ls --color=none -a ${repoLocation}" 
  "ls --color=none ${repoLocation}/venv/bin/activate" 
)

for cmd in "${commands[@]}"; do
  echo "### Command: $cmd"
  bash -c "$cmd"
done

## Virtual Environment Recovery

✅ Expectation
Your terminal prompt shows (venv) prefix.

### Run Vitual Environment

```bash
cd ~/jm1021/student
source venv/bin/activate
```

❌ If it fails

```bash
./scripts/venv.sh
source venv/bin/activate
```

### VSCode Launch and Memories

✅ Satisfying the pre-requisites

- In project directory of your repo `pwd`
- Sourcing virtual environment `source venv/bin/activate`
- Ensure your terminal prompt shows the active virtual environment `(venv)`.

You are now ready to load VSCode and build a proper memory to open your project.

```bash
code .
```

✅ Verify VSCode launch

- Terminal and presence of `(venv)` prompt
- Open your Jokes IPYNB notebook and select the Python kernel with the `venv` prefix.

❌ If you fail verification

You may have opened your repo project without activating the proper `(venv)` environment.

Check the `Recent` listings. If there are entries that look incorrect or outdated (bad memories), remove them all.

- Shift-Cmd-P (Mac) or Shift-Ctl-P (Windows, KASM)
- type: Clear Recently Open -- select and confirm
- close VSCode
- Repeat VSCode Launch and Memories

---

## Version Checks

✅ Expectation: Version output for all tools 

In VSCode open this file (tool_setup-trouble.ipynb),  run the script with comment "Version and configuration checks".

- Output is expected for each `### Command`
- Version may be slightly different, but ask if you are not sure
* Java and IJava kernels are required for CSA only

❌ If anything fails

Best course of action is to run the specific OS specific activate scripts from `opencs/pages` project

WSL or Linux Users:  Ubuntu, Kali, or Mint

```bash
./scripts/activate_ubuntu.sh # ubuntu script works for all 3 variants, as they use same apt package manager
```

MacOS users

```bash
./scripts/activate_macos.sh # macos
```

Problems with git config

```bash
./scripts/activate.sh # help setup git config options
```


In [ ]:
%%script bash

# Version and configuration checks 
commands=(
  "python --version" 
  "pip --version" 
  "ruby -v" 
  "bundle -v" 
  "gem -v" 
  "jupyter --version" 
  "jupyter kernelspec list" 
  "git config --global user.name" 
  "git config --global user.email"
)

for cmd in "${commands[@]}"; do
  echo "### Command: $cmd"
  bash -c "$cmd"
done

### Command: python --version
Python 3.14.0
### Command: pip --version
pip 25.3 from /Users/johnmortensen/opencs/pages/venv/lib/python3.14/site-packages/pip (python 3.14)
### Command: ruby -v
ruby 3.4.7 (2025-10-08 revision 7a5688e2a2) +PRISM [arm64-darwin25]
### Command: bundle -v
Bundler version 2.7.1
### Command: gem -v
3.7.2
### Command: jupyter --version
Selected Jupyter core packages...
IPython          : 9.7.0
ipykernel        : 7.1.0
ipywidgets       : not installed
jupyter_client   : 8.6.3
jupyter_core     : 5.9.1
jupyter_server   : 2.17.0
jupyterlab       : 4.5.0
nbclient         : 0.10.2
nbconvert        : 7.16.6
nbformat         : 5.10.4
notebook         : 7.5.0
qtconsole        : not installed
traitlets        : 5.14.3
### Command: jupyter kernelspec list
Available kernels:
  java           /Users/johnmortensen/Library/Jupyter/kernels/java
  jbang-ijava    /Users/johnmortensen/Library/Jupyter/kernels/jbang-ijava
  python3        /Users/johnmortensen/opencs/pages/venv/share

## 🖥️ Machine Verification

Check that your machine has all required tools installed and configured correctly.
Start `host.py` before clicking the button below.

<button id="mv-run-btn" onclick="mvRun()">▶ Run Verification</button>

<div id="mv-error" style="display:none;margin-top:18px;background:#1f1215;border:1px solid #6e2930;border-radius:8px;padding:16px 20px;">
  <strong style="color:#f85149;">⚠ Could not connect to host.py</strong>
  <p style="font-size:0.85rem;color:#c9d1d9;margin-top:8px;">Could not reach <code>localhost:6741/api/host</code>. Start the local server first:</p>
  <ol style="font-size:0.82rem;color:#8b949e;line-height:1.8;">
    <li>Open a terminal and navigate to the <code>flask</code> project folder</li>
    <li>Run: <code>./scripts/venv.sh</code> then <code>source venv/bin/activate</code></li>
    <li>Run: <code>python host.py</code></li>
    <li>Return here and click <strong>Run Verification</strong> again</li>
  </ol>
</div>

<div id="mv-results" style="display:none;margin-top:24px;"></div>

<div id="mv-summary" style="display:none;margin-top:16px;padding:12px 18px;background:#161b22;border:1px solid #30363d;border-radius:8px;font-family:monospace;font-size:0.85rem;">
  <span id="mv-count-pass" style="color:#3fb950;"></span>
  &nbsp;&nbsp;
  <span id="mv-count-fail" style="color:#f85149;"></span>
</div>

<!-- ── Progress Graph ─────────────────────────────────────── -->
<div id="prog-section" style="margin-top:32px;">
  <div style="display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:12px;margin-bottom:1rem;">
    <h3 style="font-family:monospace;font-size:0.85rem;text-transform:uppercase;letter-spacing:0.1em;color:#7d8590;margin:0;">Installation Progress</h3>
    <div style="display:flex;gap:8px;align-items:center;flex-wrap:wrap;">
      <select id="prog-chart-type" style="font-size:12px;background:#161b22;color:#c9d1d9;border:1px solid #30363d;border-radius:6px;padding:4px 8px;">
        <option value="line">Line graph</option>
        <option value="bar">Bar graph</option>
        <option value="heatmap">Heatmap</option>
      </select>
      <button id="prog-reset-btn" style="font-size:12px;background:#1f1215;color:#f85149;border:1px solid #6e2930;border-radius:6px;padding:4px 12px;cursor:pointer;">Reset history</button>
    </div>
  </div>
  <div style="display:flex;align-items:center;gap:8px;margin-bottom:1rem;flex-wrap:wrap;">
    <label style="font-family:monospace;font-size:0.78rem;color:#7d8590;">Student name:</label>
    <input id="prog-name" type="text" placeholder="Enter your name"
      style="font-family:monospace;font-size:0.78rem;background:#161b22;color:#c9d1d9;border:1px solid #30363d;border-radius:6px;padding:4px 10px;width:180px;" />
  </div>
  <div id="prog-empty" style="font-family:monospace;font-size:0.78rem;color:#7d8590;padding:24px 0;text-align:center;">
    No runs yet — click Run Verification above.
  </div>
  <div style="position:relative;width:100%;height:280px;display:none;" id="prog-chart-wrap">
    <canvas id="prog-chart" role="img" aria-label="Software installation progress over time"></canvas>
  </div>
  <div id="prog-heatmap" style="display:none;"></div>
  <table id="prog-table" style="display:none;width:100%;border-collapse:collapse;margin-top:1rem;font-family:monospace;font-size:0.75rem;">
    <thead>
      <tr>
        <th style="text-align:left;color:#7d8590;padding:4px 8px;border-bottom:1px solid #30363d;">Student</th>
        <th style="text-align:left;color:#7d8590;padding:4px 8px;border-bottom:1px solid #30363d;">Installed</th>
        <th style="text-align:left;color:#7d8590;padding:4px 8px;border-bottom:1px solid #30363d;">/ Total</th>
        <th style="text-align:left;color:#7d8590;padding:4px 8px;border-bottom:1px solid #30363d;">Timestamp</th>
      </tr>
    </thead>
    <tbody id="prog-log-body"></tbody>
  </table>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>

<script>
(function() {
  const STORE_KEY = 'mv_progress_v1';
  const TOTAL = 9;
  let log = JSON.parse(localStorage.getItem(STORE_KEY) || '[]');
  let chartInst = null;

function save() { localStorage.setItem(STORE_KEY, JSON.stringify(log)); }

  function fmtDate(iso) {
    var d = new Date(iso);
    return d.toLocaleDateString(undefined,{month:'short',day:'numeric'})
      + ' ' + d.toLocaleTimeString(undefined,{hour:'2-digit',minute:'2-digit'});
  }

  function renderAll() {
    var type = document.getElementById('prog-chart-type').value;
    var empty = document.getElementById('prog-empty');
    var wrap  = document.getElementById('prog-chart-wrap');
    var canvas= document.getElementById('prog-chart');
    var hm    = document.getElementById('prog-heatmap');
    var tbl   = document.getElementById('prog-table');
    if (log.length === 0) {
      empty.style.display='block'; wrap.style.display='none';
      hm.style.display='none'; tbl.style.display='none';
      if (chartInst) { chartInst.destroy(); chartInst = null; }
      return;
    }
    empty.style.display = 'none';
    tbl.style.display = 'table';
    if (type === 'heatmap') {
      wrap.style.display='none'; hm.style.display='block';
      if (chartInst) { chartInst.destroy(); chartInst = null; }
      renderHeatmap();
    } else {
      hm.style.display='none'; wrap.style.display='block';
      renderChart(type);
    }
    renderTable();
  }

  function renderChart(type) {
    if (chartInst) chartInst.destroy();
    chartInst = new Chart(document.getElementById('prog-chart'), {
      type: type === 'bar' ? 'bar' : 'line',
      data: {
        labels: log.map(function(e){ return fmtDate(e.ts); }),
        datasets: [{
          label: 'Installed',
          data: log.map(function(e){ return e.count; }),
          borderColor: '#3fb950',
          backgroundColor: type==='bar' ? 'rgba(63,185,80,0.6)' : 'rgba(63,185,80,0.15)',
          fill: type==='line', tension: 0.35,
          pointRadius: 5, pointBackgroundColor: '#3fb950', borderWidth: 2,
        }]
      },
      options: {
        responsive: true, maintainAspectRatio: false,
        plugins: { legend: { display: false } },
        scales: {
          y: { min:0, max:TOTAL, ticks:{ stepSize:1, color:'#7d8590' }, grid:{ color:'rgba(255,255,255,0.06)' } },
          x: { ticks:{ color:'#7d8590', maxRotation:45, autoSkip:false }, grid:{ color:'rgba(255,255,255,0.06)' } }
        }
      }
    });
  }

  function renderHeatmap() {
    var days = {};
    log.forEach(function(e) {
      var day = e.ts.slice(0,10);
      if (!days[day] || e.count > days[day]) days[day] = e.count;
    });
    var sorted = Object.keys(days).sort();
    var html = '<div style="display:flex;flex-wrap:wrap;gap:6px;padding:12px 0;">';
    sorted.forEach(function(day) {
      var v = days[day]; var alpha = 0.1 + (v/TOTAL)*0.85;
      html += '<div title="'+day+': '+v+'/'+TOTAL+'" style="width:48px;height:48px;border-radius:6px;background:rgba(63,185,80,'+alpha.toFixed(2)+');display:flex;align-items:center;justify-content:center;flex-direction:column;">'
        + '<span style="font-size:14px;font-weight:600;color:#fff;">'+v+'</span>'
        + '<span style="font-size:9px;color:rgba(255,255,255,0.7);">'+day.slice(5)+'</span></div>';
    });
    html += '</div><p style="font-size:11px;color:#7d8590;">Best run per day. Darker = more installed.</p>';
    document.getElementById('prog-heatmap').innerHTML = html;
  }

  function renderTable() {
    document.getElementById('prog-log-body').innerHTML = log.slice().reverse().map(function(e) {
      return '<tr>'
        + '<td style="padding:3px 8px;color:#c9d1d9;border-bottom:1px solid #21262d;">'+e.student+'</td>'
        + '<td style="padding:3px 8px;color:#3fb950;border-bottom:1px solid #21262d;">'+e.count+'</td>'
        + '<td style="padding:3px 8px;color:#7d8590;border-bottom:1px solid #21262d;">'+e.total+'</td>'
        + '<td style="padding:3px 8px;color:#7d8590;border-bottom:1px solid #21262d;">'+fmtDate(e.ts)+'</td>'
        + '</tr>';
    }).join('');
  }

  document.getElementById('prog-chart-type').addEventListener('change', renderAll);
  document.getElementById('prog-reset-btn').addEventListener('click', function() {
    if (!confirm('Clear all saved progress history?')) return;
    log = []; save(); renderAll();
  });

  // Called by mvRun() after verification completes
  window.__mvRecordRun = function(passCount) {
    var name = document.getElementById('prog-name').value.trim() || 'Anonymous';
    var entry = { student: name, count: passCount, total: TOTAL, ts: new Date().toISOString() };
    log.push(entry);
    save();
    renderAll();
  };

  renderAll();
})();
</script>

<script>
async function mvRun() {
  var btn     = document.getElementById('mv-run-btn');
  var errBox  = document.getElementById('mv-error');
  var results = document.getElementById('mv-results');
  var summary = document.getElementById('mv-summary');

  errBox.style.display  = 'none';
  results.style.display = 'none';
  summary.style.display = 'none';
  btn.disabled = true;
  btn.textContent = 'Checking...';

  var data;
  try {
    var res = await fetch('http://localhost:6741/api/host', { signal: AbortSignal.timeout(8000) });
    if (!res.ok) throw new Error('bad response');
    data = await res.json();
  } catch(e) {
    errBox.style.display = 'block';
    btn.disabled = false;
    btn.textContent = '▶ Run Verification';
    return;
  }

  function card(label, toolObj) {
    if (!toolObj) return '';
    var installed = toolObj.installed;
    var value = toolObj.version || toolObj.value || '';
    var display = value ? (value.length > 38 ? value.slice(0,38)+'...' : value) : (installed ? 'installed' : 'not found');
    var color = installed ? '#3fb950' : '#f85149';
    var icon  = installed ? '✅' : '❌';
    return '<div style="display:flex;align-items:center;gap:10px;background:#161b22;border:1px solid #30363d;border-left:3px solid '+color+';border-radius:8px;padding:10px 14px;margin-bottom:8px;">'
      + '<span>' + icon + '</span>'
      + '<div>'
      + '<div style="font-family:monospace;font-size:0.8rem;font-weight:600;color:#e6edf3;">' + label + '</div>'
      + '<div style="font-family:monospace;font-size:0.73rem;color:'+color+';">' + display + '</div>'
      + '</div></div>';
  }

  var html = '<p style="font-family:monospace;font-size:0.75rem;color:#7d8590;margin-bottom:12px;">💻 '
    + (data.OS||'') + ' ' + (data['OS Release']||'') + ' · ' + (data.Architecture||'') + '</p>';

  html += '<p style="font-family:monospace;font-size:0.72rem;text-transform:uppercase;letter-spacing:0.1em;color:#7d8590;margin:12px 0 6px;">Git</p>';
  html += card('git', data.git) + card('git user.name', data.git_user_name) + card('git user.email', data.git_user_email);

  html += '<p style="font-family:monospace;font-size:0.72rem;text-transform:uppercase;letter-spacing:0.1em;color:#7d8590;margin:12px 0 6px;">Python & Pip</p>';
  html += card('Python', data.python) + card('pip', data.pip);

  html += '<p style="font-family:monospace;font-size:0.72rem;text-transform:uppercase;letter-spacing:0.1em;color:#7d8590;margin:12px 0 6px;">Ruby & Bundler</p>';
  html += card('Ruby', data.ruby) + card('Bundler', data.bundler) + card('RubyGems', data.gem);

  html += '<p style="font-family:monospace;font-size:0.72rem;text-transform:uppercase;letter-spacing:0.1em;color:#7d8590;margin:12px 0 6px;">Jupyter</p>';
  html += card('Jupyter CLI', data.jupyter);
  if (data.jupyter_kernelspecs && data.jupyter_kernelspecs.installed) {
    (data.jupyter_kernelspecs.value||'').split('\n').filter(function(l){return l.includes('/');}).forEach(function(line){
      var kname = line.trim().split(/\s+/)[0] || 'kernel';
      html += card('kernel: ' + kname, { installed: true, value: 'registered' });
    });
  }

  if (data.OS === 'Darwin') {
    html += '<p style="font-family:monospace;font-size:0.72rem;text-transform:uppercase;letter-spacing:0.1em;color:#7d8590;margin:12px 0 6px;">macOS Tools</p>';
    html += card('Homebrew', data.brew) + card('Xcode CLI Tools', data.xcode_select);
  }

  results.innerHTML = html;
  results.style.display = 'block';

  var keys = ['git','git_user_name','git_user_email','python','pip','ruby','bundler','gem','jupyter'];
  if (data.OS === 'Darwin') keys.push('brew','xcode_select');
  var pass = 0, fail = 0;
  keys.forEach(function(k){ if(data[k]){ data[k].installed ? pass++ : fail++; } });

  document.getElementById('mv-count-pass').textContent = pass + ' passing';
  document.getElementById('mv-count-fail').textContent = fail > 0 ? fail + ' failing' : '';
  summary.style.display = 'block';
  
  // ✅ ADD THIS
  if (window.__mvRecordRun) {
    window.__mvRecordRun(pass);
  }
  
  btn.disabled = false;
  btn.textContent = '▶ Run Again';

}
</script>